# Stock Price Prediction with LSTM

In this exercise we use an **LSTM** to predict Apple stock prices from historical data. We also discuss the inherent limitations of applying machine learning to stock prediction.

**Dataset:** `apple_stock.csv` -- Daily closing prices of Apple (AAPL), January 2015 to December 2023 (2,264 trading days). The prices are adjusted for splits and dividends; the file has a single `Close` column and two extra header rows from its export.

**Learning objectives:**
1. Load and visualize historical stock price data.
2. Prepare sequential data for LSTM prediction, with the scaler fitted on the training days only.
3. Build and train an LSTM model.
4. Evaluate predictions on a held-out test period against persistence (yesterday's close) on the same days.
5. Evaluate daily changes, not only price levels, and understand the limitations of ML-based stock prediction.

**Evaluation design.** Windows are split by target day; the last 20% form the test period, used once at the end. The number of epochs is fixed in advance, not tuned; to tune it, carve a validation period out of the training days, never out of the test days. This is a method demonstration, not evidence of trading value. Runtime: about 1.5 minutes on an Apple M3 Max laptop CPU; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# The export has two extra header rows ("Ticker,AAPL" and "Date,") below the column names
df = pd.read_csv("apple_stock.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
df["Date"] = pd.to_datetime(df["Date"])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df.describe()

In [ ]:
# Check date range
print(f"Date range: {df['Date'].iloc[0]:%Y-%m-%d} to {df['Date'].iloc[-1]:%Y-%m-%d}")
print(f"Total trading days: {len(df)}")
print(f"Missing values: {df.isnull().sum().sum()}")

---
## 3. Visualize Stock Price Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Close price
axes[0].plot(df["Date"], df["Close"], linewidth=0.8)
axes[0].set_title("Apple Stock - Close Price")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True, alpha=0.3)

# Daily change (the file has no volume column)
axes[1].plot(df["Date"], df["Close"].diff(), linewidth=0.5)
axes[1].set_title("Apple Stock - Daily Change in Close Price")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Change ($)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Prepare Data: Close Price, Normalize, Create Sequences

Each window belongs to the period that contains its **target** day: the first 80% of windows for training, the last 20% for the test. Test windows may read the preceding 60 observed days as inputs, which is what a deployed forecast sees, but no training target lies in the test period. The scaler is fitted on the days used by the training windows only; test prices above the training range then map above 1, which is expected.

In [ ]:
# Use Close price as our target
close_prices = df["Close"].values.reshape(-1, 1).astype(np.float32)

# Normalize to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
# Match the later 80/20 chronological split of one-step windows.
SEQ_LENGTH = 60
train_sequence_count = int((len(close_prices) - SEQ_LENGTH) * 0.8)
training_raw_end = SEQ_LENGTH + train_sequence_count
scaler.fit(close_prices[:training_raw_end])  # training days only
close_scaled = scaler.transform(close_prices)
# Later observed values may fall outside [0, 1]; do not refit or clip them.

print(f"Scaler fitted on days 1-{training_raw_end} of {len(close_prices)}")
print(f"Original range: [{close_prices.min():.2f}, {close_prices.max():.2f}]")
print(f"Scaled range:   [{close_scaled.min():.4f}, {close_scaled.max():.4f}]")

In [ ]:
def create_sequences(data, seq_length):
    """Create sliding-window sequences for time series prediction."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


SEQ_LENGTH = 60  # use past 60 trading days (~3 months) to predict next day

X, y = create_sequences(close_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Chronological split by target day: 80% train, 20% test
split = int(len(X) * 0.8)
assert split + SEQ_LENGTH == training_raw_end  # the scaler saw training days only
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Test target days: {df['Date'].iloc[split + SEQ_LENGTH]:%Y-%m-%d} to {df['Date'].iloc[-1]:%Y-%m-%d}")

---
## 5. Build LSTM Model

In [ ]:
class StockLSTM(nn.Module):
    """LSTM for stock price prediction."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        return self.fc(last_hidden)


model = StockLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 6. Train the Model

`NUM_EPOCHS = 100` is fixed in advance; it is not chosen by looking at the test days.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 100
train_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(X_train_t)
    train_losses.append(epoch_loss)

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  Loss: {epoch_loss:.6f}")

---
## 7. Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Predict Future Prices and Plot Against Actual

This is the final evaluation: the test days are used once, for the model trained above.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t.to(DEVICE)).cpu().numpy()

# Inverse transform
y_pred = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test)

rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mae = mean_absolute_error(y_actual, y_pred)
print(f"Test RMSE: ${rmse:.2f}")
print(f"Test MAE:  ${mae:.2f}")

In [ ]:
# Plot full price history with prediction overlay
plt.figure(figsize=(14, 6))

# All actual prices
plt.plot(range(len(close_prices)), close_prices, label="Actual", linewidth=0.8)

# Predicted prices (offset by train length + seq_length)
pred_start = split + SEQ_LENGTH
pred_x = range(pred_start, pred_start + len(y_pred))
plt.plot(pred_x, y_pred, label="Predicted", linewidth=1.2, color="red", alpha=0.8)

plt.axvline(x=pred_start, color="gray", linestyle="--", alpha=0.5, label="Train/Test Split")
plt.title("Apple Stock Price: Actual vs Predicted")
plt.xlabel("Trading Day")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in view of test period
plt.figure(figsize=(14, 5))
plt.plot(y_actual, label="Actual", marker=".", markersize=2, linewidth=0.8)
plt.plot(y_pred, label="Predicted", marker=".", markersize=2, linewidth=0.8)
plt.title("Test Period: Actual vs Predicted")
plt.xlabel("Test Day")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Limitations of Stock Prediction with ML

While our LSTM may appear to produce reasonable predictions, it is important to understand the fundamental limitations:

**1. The Efficient Market Hypothesis (EMH):**
Stock prices reflect all available information. Any predictable pattern would be quickly arbitraged away by market participants.

**2. The Persistence (Copy) Illusion:**
When the model appears to "predict" well, it is often because tomorrow's price is usually close to today's price. The model may simply be learning to output a value close to its most recent input (a naive lag-1 persistence forecast). On a long price chart the one-day lag is invisible; compare with persistence on the same days and evaluate daily changes (below).

**3. Look-Ahead Bias:**
Using information that was not available at the forecast origin -- for example a scaler fitted on the whole series, a random split that puts later days into training, or features computed from future prices. This notebook fits the scaler on the training days only and splits by target day. One subtle source remains: adjusted prices are recomputed after later splits and dividends, so the early values in the file are not the prices that were quoted at the time.

**4. Non-Stationarity:**
Financial time series are non-stationary -- their statistical properties change over time. A model trained on one regime may fail in another.

**5. External Factors:**
Stock prices are influenced by earnings reports, macroeconomic events, geopolitical developments, and sentiment -- none of which are captured by historical price alone.

**6. Practical Implications:**
Even small prediction errors can lead to large financial losses when leveraged. The real question is not "can we predict direction?" but "can we predict direction better than chance, after accounting for transaction costs?"

In [ ]:
# Demonstrate the naive baseline: just predict yesterday's price
naive_pred = scaler.inverse_transform(X_test[:, -1, :])  # last value in each window
naive_rmse = np.sqrt(mean_squared_error(y_actual, naive_pred))
naive_mae = mean_absolute_error(y_actual, naive_pred)

print("Comparison with Naive Baseline (predict yesterday's close), same test days:")
print(f"  Naive RMSE: ${naive_rmse:.2f}   MAE: ${naive_mae:.2f}")
print(f"  LSTM  RMSE: ${rmse:.2f}   MAE: ${mae:.2f}")

if rmse < naive_rmse:
    print(f"\n  LSTM improves over naive by {(naive_rmse - rmse) / naive_rmse * 100:.1f}% RMSE")
else:
    print(f"\n  LSTM does NOT beat the naive baseline -- a common outcome!")

### Evaluate daily changes, not only price levels

A forecast of tomorrow's *level* can look excellent simply because tomorrow's price is close to today's. What a trader or planner needs is the *change*. Convert both forecasts to changes from the last observed close at each origin; persistence then predicts a change of zero. An R^2 near or below zero on changes means no skill, however good the level plot looks. For the direction of the move, compare with always predicting "up".

In [ ]:
# Changes relative to the last observed close at each forecast origin (same test days)
last_close = naive_pred.ravel()
actual_change = y_actual.ravel() - last_close
lstm_change = y_pred.ravel() - last_close

print(f"R^2 on price levels:  LSTM {r2_score(y_actual, y_pred):.3f}   "
      f"persistence {r2_score(y_actual, naive_pred):.3f}")
print(f"R^2 on daily changes: LSTM {r2_score(actual_change, lstm_change):.3f}   "
      f"persistence {r2_score(actual_change, np.zeros_like(actual_change)):.3f}")
print(f"Direction accuracy:   LSTM {np.mean(np.sign(lstm_change) == np.sign(actual_change)):.1%}   "
      f"always 'up' {np.mean(actual_change > 0):.1%}")
print(f"The LSTM predicts a rise on {np.mean(lstm_change > 0):.0%} of the test days")

---
## 10. Summary

In this notebook we:

1. Loaded Apple stock price data and visualized the close price and its daily changes.
2. Normalized close prices with a `MinMaxScaler` fitted on the training days and created sliding-window sequences.
3. Built a 2-layer LSTM with an FC output head.
4. Trained the model for a fixed number of epochs and plotted the loss curve.
5. Predicted the test days once and plotted predictions against actual prices.
6. Compared against a naive baseline (previous day's close) on the same days, on price levels and on daily changes.
7. Discussed the fundamental limitations of ML-based stock prediction.

**Key takeaways:**
- LSTMs can learn patterns in sequential data, but stock prices are notoriously difficult to predict.
- Always compare against a naive baseline -- many "good" stock models are actually just learning lag-1 persistence.
- Judge the forecast on daily changes: a high R^2 on price levels is mostly the persistence of the price.
- This exercise demonstrates the *mechanics* of applying LSTMs to financial data, not a viable trading strategy.
- Real quantitative finance models incorporate many additional features, risk management, and rigorous backtesting.